# Chapter 2 — Vector Spaces and Duality

Companion notebook to Chapter 2 of *Advanced Calculus for Machine Learning*.
It checks the numerical claims of the chapter with PyTorch. Everything runs
on a CPU in a few seconds.

Conventions of the chapter: vectors are **columns**, covectors are **rows**,
and $A^\top$ is the dual operator (no inner product is used anywhere in this
notebook, except where a library routine needs one internally to compute a
null space).

Contents:

1. The dual basis is given by the rows of $E^{-1}$
2. Change of basis: columns by $P^{-1}$, rows by $P$
3. The dual operator is the vector–Jacobian product
4. The matrix of the dual operator is the transpose, in any pair of bases
5. Quadrature weights are the components of a covector (Exercise 2.2)
6. Rank, annihilators, and what a layer cannot see
7. Three kinds of square arrays
8. The basis test: gradient descent, sign descent, Adam
9. Sharpness depends on the basis

In [1]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. The dual basis is given by the rows of $E^{-1}$

Proposition 2.10 and Exercise 2.1: if the columns of $E$ are the basis
vectors, the rows of $E^{-1}$ are the dual basis forms.

In [2]:
E = torch.tensor([[1.0, 1.0, 1.0],
                  [0.0, 1.0, 1.0],
                  [0.0, 0.0, 1.0]])      # columns e1, e2, e3
E_dual = torch.linalg.inv(E)             # rows e1*, e2*, e3*
print("dual basis (rows):\n", E_dual)
assert torch.allclose(E_dual @ E, torch.eye(3))      # e_i*(e_j) = delta_ij

v = torch.tensor([3.0, 2.0, 1.0])
phi = torch.tensor([1.0, 1.0, 1.0])      # the form x1 + x2 + x3, as a row
v_comp = E_dual @ v                      # components of v in the basis e
phi_comp = phi @ E                       # components of phi: its values on e_j
print("[v]   =", v_comp.tolist())
print("[phi] =", phi_comp.tolist())
print("[phi][v] =", (phi_comp @ v_comp).item(), "  phi(v) =", (phi @ v).item())
assert torch.equal(v_comp, torch.tensor([1.0, 1.0, 1.0]))
assert torch.equal(phi_comp, torch.tensor([1.0, 2.0, 3.0]))

dual basis (rows):
 tensor([[ 1., -1.,  0.],
        [ 0.,  1., -1.],
        [ 0.,  0.,  1.]])
[v]   = [1.0, 1.0, 1.0]
[phi] = [1.0, 2.0, 3.0]
[phi][v] = 6.0   phi(v) = 6.0


The pitfall of Section 2.2: two bases with the same first vector have
different first dual forms.

In [3]:
E1 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])    # e  = ((1,0), (0,1))
E2 = torch.tensor([[1.0, 1.0], [0.0, 1.0]])    # e' = ((1,0), (1,1))
print("e1* :", torch.linalg.inv(E1)[0].tolist(), "   e'1* :", torch.linalg.inv(E2)[0].tolist())
print("e2* :", torch.linalg.inv(E1)[1].tolist(), "   e'2* :", torch.linalg.inv(E2)[1].tolist())

e1* : [1.0, 0.0]    e'1* : [1.0, -1.0]
e2* : [0.0, 1.0]    e'2* : [0.0, 1.0]


## 2. Change of basis

Propositions 2.2 and 2.8: with $e'_j = \sum_iP_{ij}e_i$, columns of
components are multiplied by $P^{-1}$, rows by $P$, matrices of maps become
$Q^{-1}[A]P$, and the pairing does not change.

In [4]:
n, m = 4, 3
P = torch.randn(n, n)
Q = torch.randn(m, m)
x, phi_row, A = torch.randn(n), torch.randn(n), torch.randn(m, n)
psi_row = torch.randn(m)

x_new = torch.linalg.solve(P, x)          # P^{-1} x
phi_new = phi_row @ P                     # row times P
A_new = torch.linalg.solve(Q, A @ P)      # Q^{-1} A P
psi_new = psi_row @ Q

assert torch.allclose(phi_new @ x_new, phi_row @ x)                  # pairing
assert torch.allclose(A_new @ x_new, torch.linalg.solve(Q, A @ x))   # [Av]'
assert torch.allclose(psi_new @ A_new, (psi_row @ A) @ P)            # [A^T psi]'
print("pairing, [Av] and [A^T psi] transform consistently")

# Example 2.9: metres -> centimetres
P_units = torch.tensor([[1 / 100]])
v_m, price_per_m = torch.tensor([3.0]), torch.tensor([2.0])
v_cm = torch.linalg.solve(P_units, v_m)
price_per_cm = price_per_m @ P_units
print("length:", v_m.item(), "m =", v_cm.item(), "cm;  price:",
      price_per_m.item(), "per m =", price_per_cm.item(), "per cm;  total:",
      (price_per_m @ v_m).item(), "=", (price_per_cm @ v_cm).item())

pairing, [Av] and [A^T psi] transform consistently
length: 3.0 m = 300.0 cm;  price: 2.0 per m = 0.02 per cm;  total: 6.0 = 6.0


## 3. The dual operator is the vector–Jacobian product

For a linear layer $z = Wx$ and a covector $\delta$ on the output space, the
pull-back $W^\top\delta$ has the row of components $[\delta][W]$. This is
what `torch.autograd.grad(..., grad_outputs=delta)` and `torch.func.vjp`
return.

In [5]:
W = torch.randn(3, 5)
x = torch.randn(5, requires_grad=True)
delta = torch.randn(3)

z = W @ x
(pulled_back,) = torch.autograd.grad(z, x, grad_outputs=delta)
assert torch.allclose(pulled_back, delta @ W)
print("autograd.grad with grad_outputs=delta  ==  delta @ W")

_, vjp_fn = torch.func.vjp(lambda t: W @ t, x.detach())
assert torch.allclose(vjp_fn(delta)[0], delta @ W)
print("torch.func.vjp                         ==  delta @ W")

# (BA)^T = A^T B^T: the backward pass goes through the layers in reverse order
A_, B_ = torch.randn(4, 5), torch.randn(3, 4)
chi = torch.randn(3)
(through_both,) = torch.autograd.grad(B_ @ (A_ @ x), x, grad_outputs=chi)
assert torch.allclose(through_both, (chi @ B_) @ A_)
print("pull-back through B after A  ==  (chi @ B) @ A")

autograd.grad with grad_outputs=delta  ==  delta @ W


torch.func.vjp                         ==  delta @ W
pull-back through B after A  ==  (chi @ B) @ A


Exercise 2.4(3): for the map $W\mapsto Wx$ (the layer as a function of its
weights), the pull-back of $\delta$ is the form with array
$[\delta]^\top x^\top$, the outer product familiar from backpropagation.

In [6]:
Wp = torch.randn(3, 5, requires_grad=True)
xv = torch.randn(5)
(G,) = torch.autograd.grad(Wp @ xv, Wp, grad_outputs=delta)
assert torch.allclose(G, torch.outer(delta, xv))
print("pull-back to the weights == outer(delta, x)")

pull-back to the weights == outer(delta, x)


## 4. The matrix of the dual operator is the transpose, in any bases

Theorem 2.13 does not require the standard bases. Take arbitrary bases $e$
of $V$ (columns of `E`) and $f$ of $W$ (columns of `F`). The matrix of $A$
in these bases is $F^{-1}AE$. We compute the matrix of $A^\top$ in the dual
bases directly from the definition (column $k$ = components of
$A^\top f_k^*$ in $e^*$ = values of $f_k^*\circ A$ on the $e_j$) and compare.

In [7]:
n, m = 4, 3
A = torch.randn(m, n)             # the map, in standard coordinates
E = torch.randn(n, n)             # basis e of V
F = torch.randn(m, m)             # basis f of W
A_ef = torch.linalg.solve(F, A @ E)             # [A] in the bases e, f

F_dual = torch.linalg.inv(F)                    # rows: f_1*, ..., f_m*
dual_matrix = torch.zeros(n, m)
for k in range(m):
    pulled = F_dual[k] @ A                      # the form f_k* o A, as a row
    dual_matrix[:, k] = pulled @ E              # its components in e*
assert torch.allclose(dual_matrix, A_ef.T)
print("matrix of the dual operator in the dual bases == transpose of [A]")

matrix of the dual operator in the dual bases == transpose of [A]


## 5. Quadrature weights are the components of a covector

Exercise 2.2: on polynomials of degree at most 2, the evaluations at
$0, 1, 2$ form a basis of the dual space, and the components of
$I(p) = \int_0^2p$ in that basis are Simpson's weights
$\tfrac13, \tfrac43, \tfrac13$.

In [8]:
nodes = torch.tensor([0.0, 1.0, 2.0])
# monomial basis (1, t, t^2): evaluation forms are the rows of the Vandermonde matrix
V = torch.stack([nodes ** k for k in range(3)], dim=1)   # V[i, k] = nodes[i]^k
I_row = torch.tensor([2.0, 2.0, 8.0 / 3.0])              # I(1), I(t), I(t^2) on [0, 2]
# I = sum_i w_i eps_i  <=>  I_row = w @ V
weights = torch.linalg.solve(V.T, I_row)
print("weights:", weights.tolist())
assert torch.allclose(weights, torch.tensor([1 / 3, 4 / 3, 1 / 3]))

coef = torch.randn(3)                                    # a random quadratic
exact = (coef * I_row).sum()
simpson = (weights * (V @ coef)).sum()
assert torch.allclose(exact, simpson)
print("integral of a random quadratic:", exact.item(), " Simpson:", simpson.item())

weights: [0.33333333333333326, 1.3333333333333335, 0.33333333333333326]
integral of a random quadratic: 2.51582289564345  Simpson: 2.5158228956434505


## 6. Rank, annihilators, and what a layer cannot see

Theorem 2.21 and Corollary 2.22: $A$ and $A^\top$ have the same rank, and
$Av = w$ is solvable exactly when every form of $\ker A^\top$ vanishes on
$w$. Exercise 2.6 uses the $3\times2$ matrix below, for which
$\ker A^\top$ is spanned by the row $[1\ {-1}\ 1]$.

In [9]:
A = torch.tensor([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
print("rank A =", torch.linalg.matrix_rank(A).item(),
      "  rank A^T =", torch.linalg.matrix_rank(A.T).item())

psi = torch.tensor([1.0, -1.0, 1.0])
assert torch.allclose(psi @ A, torch.zeros(2))           # psi is in ker A^T

for w in (torch.tensor([1.0, 3.0, 2.0]), torch.tensor([1.0, 0.0, 0.0])):
    sol = torch.linalg.lstsq(A, w.unsqueeze(1)).solution.squeeze(1)
    solvable = torch.allclose(A @ sol, w)
    print(f"w = {w.tolist()}:  psi(w) = {(psi @ w).item():+.1f}   solvable: {solvable}")

rank A = 2   rank A^T = 2
w = [1.0, 3.0, 2.0]:  psi(w) = +0.0   solvable: True
w = [1.0, 0.0, 0.0]:  psi(w) = +1.0   solvable: False


A loss of the form $L(\theta) = h(X\theta)$ is constant along $\ker X$, so
the covector computed by `backward()` vanishes on $\ker X$: it belongs to
$(\ker X)^\circ = \operatorname{im}X^\top$, the span of the rows of $X$.
Here there are more parameters ($p = 6$) than data points ($n = 3$).

In [10]:
n_data, p = 3, 6
X = torch.randn(n_data, p)
y = torch.randn(n_data)
theta = torch.randn(p, requires_grad=True)
loss = torch.nn.functional.smooth_l1_loss(torch.tanh(X @ theta), y)
loss.backward()
g = theta.grad                                           # components of a covector

# a basis of ker X (the SVD is only a convenient way to get one)
_, S, Vh = torch.linalg.svd(X)
kernel = Vh[n_data:].T                                   # p x (p - n) matrix, X @ kernel = 0
assert torch.allclose(X @ kernel, torch.zeros(n_data, p - n_data), atol=1e-12)
print("values of the covector on a basis of ker X:", (g @ kernel).tolist())
assert torch.allclose(g @ kernel, torch.zeros(p - n_data), atol=1e-12)

# equivalently: g is a combination of the rows of X
rank_X = torch.linalg.matrix_rank(X).item()
rank_aug = torch.linalg.matrix_rank(torch.cat([X, g.unsqueeze(0)])).item()
print("rank of X:", rank_X, "  rank of X with the row g appended:", rank_aug)
assert rank_X == rank_aug

values of the covector on a basis of ker X: [-5.4293745779757815e-17, -5.923201891169696e-17, -5.701169795345748e-17]
rank of X: 3   rank of X with the row g appended: 3


## 7. Three kinds of square arrays

Proposition 2.17: the same array $M$ becomes $P^{-1}MP$, $P^\top MP$ or
$P^{-1}MP^{-\top}$ according to what it represents. Only in the first case
are the eigenvalues preserved (Exercise 2.9).

In [11]:
M = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
P = torch.tensor([[1.0, 1.0], [0.0, 1.0]])
P_inv = torch.linalg.inv(P)

as_operator = P_inv @ M @ P
as_bilinear = P.T @ M @ P
as_dual_to_primal = P_inv @ M @ P_inv.T
for name, arr in [("operator        P^-1 M P   ", as_operator),
                  ("bilinear form   P^T M P    ", as_bilinear),
                  ("map V* -> V     P^-1 M P^-T", as_dual_to_primal)]:
    eig = torch.linalg.eigvals(arr).real.sort().values
    print(name, arr.tolist(), " eigenvalues:", [round(e, 4) for e in eig.tolist()])

assert torch.equal(as_operator, torch.tensor([[2.0, 1.0], [0.0, 1.0]]))
assert torch.equal(as_bilinear, torch.tensor([[2.0, 2.0], [2.0, 3.0]]))
assert torch.equal(as_dual_to_primal, torch.tensor([[3.0, -1.0], [-1.0, 1.0]]))

operator        P^-1 M P    [[2.0, 1.0], [0.0, 1.0]]  eigenvalues: [1.0, 2.0]
bilinear form   P^T M P     [[2.0, 2.0], [2.0, 3.0]]  eigenvalues: [0.4384, 4.5616]
map V* -> V     P^-1 M P^-T [[3.0, -1.0], [-1.0, 1.0]]  eigenvalues: [0.5858, 3.4142]


Proposition 2.18 and Exercise 2.7: the identification $V\to V^*$ attached
to the basis $e' = ((1,0),(1,1))$ has the array $P^{-\top}P^{-1}$ in the
standard basis, which is not the identity.

In [12]:
G = P_inv.T @ P_inv
print("array of b_e' in the basis e:\n", G)
assert torch.equal(G, torch.tensor([[1.0, -1.0], [-1.0, 2.0]]))
assert torch.allclose(P.T @ G @ P, torch.eye(2))     # it is the identity in e'

array of b_e' in the basis e:
 tensor([[ 1., -1.],
        [-1.,  2.]])


## 8. The basis test

A loss is trained in the coordinates $x' = P^{-1}x$ of another basis: the
optimizer sees $L'(x') = L(Px')$. We take one step there and express the
result in the original coordinates. Proposition 2.24 predicts
$x - \eta PP^\top\gamma$ for gradient descent on raw partial derivatives.

In [13]:
def loss_fn(x):
    return torch.log(1 + torch.exp(x[0] - 2 * x[1])) + 0.3 * (x ** 2).sum() + x[0] * x[1]


def step_in_basis(make_opt, x0, P, steps=1):
    # run `steps` optimizer steps in the coordinates of the basis P; return old coordinates
    xp = torch.linalg.solve(P, x0).clone().requires_grad_(True)
    opt = make_opt([xp])
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(P @ xp).backward()
        opt.step()
    return P @ xp.detach()


x0 = torch.tensor([0.7, -0.4])
eta = 0.1
xg = x0.clone().requires_grad_(True)
loss_fn(xg).backward()
gamma = xg.grad.clone()                                   # partial derivatives at x0

r = 1 / math.sqrt(2)
bases = {
    "identity": torch.eye(2),
    "rotation by 45 degrees": torch.tensor([[r, -r], [r, r]]),
    "unit of e1 doubled": torch.tensor([[2.0, 0.0], [0.0, 1.0]]),
    "shear": torch.tensor([[1.0, 1.0], [0.0, 1.0]]),
    "signed permutation": torch.tensor([[0.0, -1.0], [1.0, 0.0]]),
}
sgd = lambda p: torch.optim.SGD(p, lr=eta)
reference = step_in_basis(sgd, x0, torch.eye(2))
for name, P in bases.items():
    new = step_in_basis(sgd, x0, P)
    predicted = x0 - eta * P @ P.T @ gamma
    assert torch.allclose(new, predicted)
    same = torch.allclose(new, reference)
    print(f"SGD, {name:24}: x - eta P P^T gamma confirmed;  same step as in e: {same}")

SGD, identity                : x - eta P P^T gamma confirmed;  same step as in e: True
SGD, rotation by 45 degrees  : x - eta P P^T gamma confirmed;  same step as in e: True
SGD, unit of e1 doubled      : x - eta P P^T gamma confirmed;  same step as in e: False
SGD, shear                   : x - eta P P^T gamma confirmed;  same step as in e: False
SGD, signed permutation      : x - eta P P^T gamma confirmed;  same step as in e: True


Example 2.25: expressing one feature in centimetres instead of metres is
the change of basis $P = \mathrm{diag}(100, 1)$ in parameter space, and
multiplies the step along that weight by $10^4$.

In [14]:
P_cm = torch.tensor([[100.0, 0.0], [0.0, 1.0]])
step_m = step_in_basis(sgd, x0, torch.eye(2)) - x0
step_cm = step_in_basis(sgd, x0, P_cm) - x0
print("ratio of the two steps, per coordinate:", (step_cm / step_m).tolist())
assert torch.allclose(step_cm / step_m, torch.tensor([1e4, 1.0]))

ratio of the two steps, per coordinate: [10000.000000000005, 1.0]


Sign descent (Example 2.26, Exercise 2.8) with $\gamma = (2, 1)$: the step
computed in the basis $P$ is $-\eta\,P\,\mathrm{sign}(P^\top\gamma)$.

In [15]:
gam = torch.tensor([2.0, 1.0])
for name, P in bases.items():
    gd = P @ P.T @ gam
    sg = P @ torch.sign(P.T @ gam)
    fmt = lambda t: [round(v, 4) + 0.0 for v in t.tolist()]
    print(f"{name:24}: gradient descent -eta*{fmt(gd)}   sign descent -eta*{fmt(sg)}")

assert torch.allclose(bases["rotation by 45 degrees"] @ torch.sign(bases["rotation by 45 degrees"].T @ gam),
                      torch.tensor([math.sqrt(2), 0.0]))
assert torch.equal(bases["unit of e1 doubled"] @ torch.sign(bases["unit of e1 doubled"].T @ gam),
                   torch.tensor([2.0, 1.0]))
assert torch.equal(bases["shear"] @ bases["shear"].T @ gam, torch.tensor([5.0, 3.0]))
assert torch.equal(bases["shear"] @ torch.sign(bases["shear"].T @ gam), torch.tensor([2.0, 1.0]))

identity                : gradient descent -eta*[2.0, 1.0]   sign descent -eta*[1.0, 1.0]
rotation by 45 degrees  : gradient descent -eta*[2.0, 1.0]   sign descent -eta*[1.4142, 0.0]
unit of e1 doubled      : gradient descent -eta*[8.0, 1.0]   sign descent -eta*[2.0, 1.0]
shear                   : gradient descent -eta*[5.0, 3.0]   sign descent -eta*[2.0, 1.0]
signed permutation      : gradient descent -eta*[2.0, 1.0]   sign descent -eta*[1.0, 1.0]


The entries of Table 2.2 for Adam and AdamW, checked over 50 steps: they
give the same points in two bases only for the signed permutation.
(Their behaviour under a change of *origin* was checked in the notebook of
Chapter 1.)

In [16]:
optimizers = {
    "SGD": lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD + weight decay": lambda p: torch.optim.SGD(p, lr=0.05, weight_decay=0.1),
    "Adam": lambda p: torch.optim.Adam(p, lr=0.05),
    "AdamW": lambda p: torch.optim.AdamW(p, lr=0.05, weight_decay=0.1),
}
names = ["rotation by 45 degrees", "unit of e1 doubled", "signed permutation"]
print(f"{'':20}" + "".join(f"{n:>26}" for n in names))
for oname, make_opt in optimizers.items():
    ref = step_in_basis(make_opt, x0, torch.eye(2), steps=50)
    row = []
    for n in names:
        end = step_in_basis(make_opt, x0, bases[n], steps=50)
        row.append("same" if torch.allclose(end, ref, atol=1e-9) else "different")
    print(f"{oname:20}" + "".join(f"{c:>26}" for c in row))

                        rotation by 45 degrees        unit of e1 doubled        signed permutation


SGD                                       same                 different                      same
SGD + weight decay                        same                 different                      same


Adam                                 different                 different                      same


AdamW                                different                 different                      same


## 9. Sharpness depends on the basis

The loss $L(x) = \tfrac12(x_1^2 + x_2^2)$ has Hessian array $I$. In the
basis $P = \mathrm{diag}(0.1, 1)$ the same function has the Hessian array
$P^\top IP = \mathrm{diag}(0.01, 1)$: a hundred times "flatter" in one
direction, although nothing has changed but the unit.

In [17]:
P_flat = torch.tensor([[0.1, 0.0], [0.0, 1.0]])
L = lambda x: 0.5 * (x ** 2).sum()
H = torch.autograd.functional.hessian(L, torch.zeros(2))
H_new = torch.autograd.functional.hessian(lambda xp: L(P_flat @ xp), torch.zeros(2))
print("Hessian array in e :", H.tolist())
print("Hessian array in e':", H_new.tolist())
assert torch.allclose(H_new, P_flat.T @ H @ P_flat)

Hessian array in e : [[1.0, 0.0], [0.0, 1.0]]
Hessian array in e': [[0.010000000000000002, 0.0], [0.0, 1.0]]
